In [0]:
from pathlib import Path
import pickle
from pyspark.sql import functions as F
from pyspark import StorageLevel
import json

In [0]:
# ============================================================
# Dataset Construction (Optimized with Reuse)
# ============================================================

def load_processed_features(spark, db, table_names, run_date, sample_window):
    """
    Load pre-processed user and product features.
    These already have encoded indices and normalized values.
    """
    """Load inference dataframes for a specific date."""
    user_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_user_processed_features']}
        WHERE partition_date between date_sub('{run_date}', {sample_window}) and '{run_date}' 
    """)
    
    product_df = spark.sql(f"""
        SELECT *
        FROM {db}.{table_names['mmc_product_processed_features']}
        WHERE partition_date between date_sub('{run_date}', {sample_window}) and '{run_date}' 
    """)
        
    # Cache since they'll be used in joins
    user_df.cache()
    product_df.cache()
    
    return user_df, product_df

def load_labels(spark, db, table_names, run_date, sample_window):
    """
    Load interactions and payment data.
    """    
    interactions_df = spark.sql(f"""
        SELECT user_id, 
            product_id, 
            if(details_viewed > 0, 1, 0) as label_detail_viewed,
            if(orders_initiated > 0, 1, 0) as label_order_initiated,
            if(orders_placed > 0, 1, 0) as label_order_placed,
            total_payment as label_payment,
            partition_date
        FROM {db}.{table_names['mmc_user_product_interaction']}
        WHERE partition_date BETWEEN date_sub('{run_date}', {sample_window}) AND '{run_date}'
    """)

    # Cache since they'll be used in joins
    interactions_df.cache()
    
    return interactions_df

def build_ranking_base_table(
    spark,
    interactions_df,
    user_features_processed_df,
    product_features_processed_df,
    user_id_col,
    product_id_col,
    user_cat_cols,
    user_num_cols,
    product_cat_cols,
    product_num_cols,
    partition_col='partition_date',
    alpha=1  # negative down-sampling ratio
):
    """
    Build ranking base table using pre-processed features
    """
    # Cache input dataframes
    interactions_df.cache()
    
    # Register as temp views
    interactions_df.createOrReplaceTempView("interactions")
    user_features_processed_df.createOrReplaceTempView("user_features_processed")
    product_features_processed_df.createOrReplaceTempView("product_features_processed")

    # Build feature selections from processed dataframes
    # Already have _idx and _norm columns from two-tower processing
    user_feature_cols = (
        [user_id_col + "_idx"] +
        user_cat_cols + [c + "_idx" for c in user_cat_cols] +
        user_num_cols + [c + "_norm" for c in user_num_cols]
    )
    
    product_feature_cols = (
        [product_id_col + "_idx"] +
        product_cat_cols + [c + "_idx" for c in product_cat_cols] +
        product_num_cols + [c + "_norm" for c in product_num_cols]
    )
    
    user_feature_select = ", ".join([f"uf.{c}" for c in user_feature_cols])
    product_feature_select = ", ".join([f"pf.{c}" for c in product_feature_cols])

    # Optimized query using pre-processed features
    query = f"""
        WITH positive_samples AS (
            SELECT * FROM interactions
            WHERE label_detail_viewed = 1 OR label_order_initiated = 1 OR label_order_placed = 1 
        ),
        
        negative_samples AS (
            SELECT * FROM interactions
            WHERE label_detail_viewed = 0 AND label_order_initiated = 0 AND label_order_placed = 0 
               AND rand() <= {alpha}
        ),
        
        base_downsample AS (
            SELECT * FROM positive_samples
            UNION ALL
            SELECT * FROM negative_samples
        )
        
        SELECT 
            bd.user_id,
            bd.product_id,
            bd.{partition_col},
            bd.label_detail_viewed,
            bd.label_order_initiated,
            bd.label_order_placed,
            bd.label_payment,
            {user_feature_select},
            {product_feature_select}
        FROM base_downsample bd
        INNER JOIN user_features_processed uf
            ON bd.user_id = uf.user_id
            AND bd.{partition_col} = uf.{partition_col}
        INNER JOIN product_features_processed pf
            ON bd.product_id = pf.product_id
            AND bd.{partition_col} = pf.{partition_col}
    """
    
    result = spark.sql(query)
    
    # Unpersist cached dataframes
    interactions_df.unpersist()
    
    return result


def remove_rows_with_null(df, columns):
    """Remove rows with null values efficiently."""
    if not columns:
        return df
    
    valid_columns = [col for col in columns if col in df.columns]
    
    if not valid_columns:
        return df
    
    return df.dropna(subset=valid_columns)


# ============================================================
# Metadata and Statistics
# ============================================================

def calculate_class_ratios(df, label_cols):
    """
    Calculate class imbalance ratios efficiently using single aggregation.
    """
    valid_columns = [col for col in label_cols if (col in df.columns) and (df.select(F.approx_count_distinct(col)).collect()[0][0]<=2)]
    
    if not valid_columns:
        return {}
    
    # Single aggregation pass for all labels
    agg_exprs = []
    for column in valid_columns:
        agg_exprs.extend([
            F.sum(F.col(column)).alias(f"{column}_sum"),
            F.count(F.col(column)).alias(f"{column}_count")
        ])
    
    result = df.agg(*agg_exprs).first()
    
    # Calculate ratios
    ratios = {}
    for column in valid_columns:
        ones_count = result[f"{column}_sum"] or 0
        total_count = result[f"{column}_count"] or 0
        zeros_count = total_count - ones_count
        
        if zeros_count == 0:
            ratio = float('inf') if ones_count > 0 else 0.0
        else:
            ratio = ones_count / zeros_count
            
        ratios[column] = ratio
    
    return ratios


def get_categorical_dimensions_from_processed(df, cat_cols):
    """
    Get categorical dimensions from already-processed dataframe.
    """
    # Single aggregation for all categorical columns
    agg_exprs = [F.max(F.col(col + "_idx")).alias(col) for col in cat_cols]
    result = df.agg(*agg_exprs).first()
    
    # Max index + 1 gives us the dimension (since we shifted by 1)
    cat_dims = {col: result[col] + 1 for col in cat_cols}
    
    return cat_dims


def save_ranking_metadata(df, label_cols, metadata_dir):
    """
    Save only ranking-specific metadata.
    Feature processing metadata already exists from two-tower model.
    """
    Path(metadata_dir).mkdir(parents=True, exist_ok=True)
    
    # Calculate class ratios (ranking-specific)
    class_ratios = calculate_class_ratios(df, label_cols)
    # Remove 'label_' prefix for consistency
    class_ratios = {k.replace('label_', ''): v for k, v in class_ratios.items() 
                   if k != 'label_payment'}
    
    # Save ranking-specific metadata
    pickle.dump(class_ratios, open(Path(metadata_dir) / 'class_ratios.pkl', 'wb'))
    # pickle.dump(cat_dims, open(Path(metadata_dir) / 'categorical_dimensions.pkl', 'wb'))
    
    print(f"Saved ranking-specific metadata to {metadata_dir}")
    print(f"  - Class ratios: {class_ratios}")
    # print(f"  - Categorical dimensions: {cat_dims}")
    
    return None

def determine_target_partitions(row_count, rows_per_partition=250000, min_partitions=64, max_partitions=512):
    if row_count <= 0:
        return min_partitions

    partition_count = (row_count + rows_per_partition - 1) // rows_per_partition
    return max(min_partitions, min(max_partitions, partition_count))


# ============================================================
# Main Processing Function
# ============================================================

def process_ranking_dataframe(df, 
                              user_id_col, product_id_col,
                              user_cat_cols, product_cat_cols,
                              user_num_cols, product_num_cols,
                              label_cols,
                              parquet_dir,
                              metadata_dir,
                              row_count):
    """
    Process ranking model data using pre-processed features.
    """
    # Cache for statistics collection
    df.cache()
    
    # All features are already processed, just need to organize columns
    cat_cols = user_cat_cols + product_cat_cols
    num_cols = user_num_cols + product_num_cols
        
    # Save ranking-specific metadata
    print("Saving ranking metadata...")
    save_ranking_metadata(df, label_cols, metadata_dir)
    
    # Select final columns for ranking model
    select_cols = [
        user_id_col, user_id_col + '_idx',
        product_id_col, product_id_col + '_idx',
        *user_cat_cols, *[c + "_idx" for c in user_cat_cols],
        *product_cat_cols, *[c + "_idx" for c in product_cat_cols],
        *user_num_cols, *[c + "_norm" for c in user_num_cols],
        *product_num_cols, *[c + "_norm" for c in product_num_cols],
        *label_cols
    ]
    
    df_final = df.select(*select_cols)
    
    # Write to parquet with adaptive partitioning and compression
    print("Writing processed data...")
    num_partitions = determine_target_partitions(row_count)
    print(f"Writing with {num_partitions} partitions")
    
    (df_final
     .repartition(num_partitions)
     .write
     .mode("overwrite")
     .option("compression", "snappy")
     .parquet(parquet_dir))
    
    # Unpersist
    df.unpersist()
    
    return None

In [0]:
# Get pipeline parameters
run_date = dbutils.widgets.get("run_date")
print("Input parameter: run_date {}".format(run_date))

table_config_path = dbutils.widgets.get("table_config_path")
with open(table_config_path, "r") as file:
    table_config = json.load(file)
db, table_names = table_config['DATABASE'], table_config['TABLE_NAMES']

model_config_path = dbutils.widgets.get("model_config_path")
with open(model_config_path, "r") as file:
    model_config = json.load(file)


In [0]:
# Configuration
user_id_col = 'user_id'
product_id_col = 'product_id'
partition_col = 'partition_date'

user_cat_cols = ["platform"]
user_num_cols=['store_visits_30d',
'store_visits_90d',
'pdp_views_30d',
'pdp_views_90d',
'tot_items_ordered_30d',
'tot_items_ordered_90d',
'tot_items_ordered_180d',
'app_items_ordered_30d',
'app_items_ordered_90d',
'app_items_ordered_180d',
'coupons_owned_30d',
'coupons_owned_90d',
'coupons_owned_180d',
'coupons_redeemed_30d',
'coupons_redeemed_90d',
'coupons_redeemed_180d']

product_cat_cols = ["product_type"]
product_num_cols=['card_uv_imps_30d',
'card_uv_imps_90d',
'pdp_uv_imps_30d',
'pdp_uv_imps_90d',
'tot_order_cnt_30d',
'tot_order_cnt_90d',
'app_order_cnt_30d',
'app_order_cnt_90d',
'avg_order_msrp',
'avg_order_baseprice_60d',
'avg_trans_price_30d',
'avg_trans_price_90d']

label_cols = ['label_detail_viewed', 'label_order_initiated', 'label_order_placed', 'label_payment']

user_feature_cols = user_cat_cols + user_num_cols
product_feature_cols = product_cat_cols + product_num_cols

parquet_dir = model_config['TRAIN_PARQUET_PATH']
metadata_dir = model_config['METADATA_PATH']

neg_ratio = model_config['neg_sample_ratio']
sample_window = model_config['model_sample_window']

In [0]:
# ========================================
# Step 1: Load pre-processed features
# ========================================
print("="*60)
print("Step 1: Loading pre-processed features")
print("="*60)

# Load processed user and product features
user_features_processed, product_features_processed = load_processed_features(
    spark, db, table_names, run_date, sample_window
)
print("Features loaded.")

# ========================================
# Step 2: Load interaction data
# ========================================
print("\n" + "="*60)
print("Step 2: Loading interaction data")
print("="*60)

interactions_df = load_labels(spark, db, table_names, run_date, sample_window)
print("Labels loaded.")

# ========================================
# Step 3: Build ranking base table
# ========================================
print("\n" + "="*60)
print("Step 3: Building ranking base table")
print("="*60)

base_table = build_ranking_base_table(
    spark,
    interactions_df,
    user_features_processed,
    product_features_processed,
    user_id_col,
    product_id_col,
    user_cat_cols,
    user_num_cols,
    product_cat_cols,
    product_num_cols,
    partition_col=partition_col,
    alpha=neg_ratio
)

# Remove nulls (should be minimal since features are pre-processed)
print("Removing null values...")
# Only check _idx and _norm columns since raw values might have been dropped
check_cols = (
    [c + "_idx" for c in user_cat_cols + product_cat_cols] +
    [c + "_norm" for c in user_num_cols + product_num_cols]
)
base_table = remove_rows_with_null(base_table, check_cols)
base_table = base_table.persist(StorageLevel.MEMORY_AND_DISK)
base_row_count = base_table.count()

print(f"Base table size: {base_row_count} rows")

# ========================================
# Step 4: Process and save ranking data
# ========================================
print("\n" + "="*60)
print("Step 4: Processing ranking data")
print("="*60)

process_ranking_dataframe(
    base_table,
    user_id_col, product_id_col,
    user_cat_cols, product_cat_cols,
    user_num_cols, product_num_cols,
    label_cols,
    parquet_dir,
    metadata_dir,
    base_row_count
)

# Calculate and display class ratios
class_ratios = calculate_class_ratios(base_table, label_cols)
class_ratios_display = {k.replace('label_', ''): v for k, v in class_ratios.items()}

# ========================================
# Final Summary
# ========================================
print("\n" + "="*60)
print("RANKING MODEL DATA PROCESSING COMPLETE!")
print("="*60)
print(f"Class ratios:          {class_ratios_display}")
print(f"\nData saved to:         {parquet_dir}")
print(f"Metadata saved to:     {metadata_dir}")
print("="*60)

# Unpersist any remaining cached dataframes
user_features_processed.unpersist()
product_features_processed.unpersist()
base_table.unpersist()